# Government Services RAG Pipeline — Granite Switch Intrinsics

This notebook demonstrates a conversational RAG pipeline where **all AI capabilities run through a single vLLM endpoint**. This tutorial uses the vLLM backend because the mellea intrinsics API (guardian, RAG, citations) currently supports vLLM only. Guardian, query rewrite, clarification, and citations are embedded adapters baked into the Granite Switch model checkpoint and activated via control tokens — no separate model services, no dynamic LoRA loading at inference time.

**What you'll build:** a 6-turn conversation that exercises every step of the pipeline — grounded answers with citations, clarification on ambiguous queries, early exit on unanswerable ones, and guardian blocks for out-of-scope or harmful requests.

## Before you start

1. **Install dependencies** (GPU + CUDA required):
   ```bash
   pip install "granite-switch[vllm]" mellea chromadb httpx python-dotenv jupyter
   ```
2. **Get a composed Granite Switch model.** Use a ready-made checkpoint from the [IBM Granite 4.1 collection on Hugging Face](https://huggingface.co/collections/ibm-granite/granite-41-language-models) (e.g. `ibm-granite/granite-switch-4.1-3b-preview`), or compose your own — see [`03_compose_granite_switch.ipynb`](./03_compose_granite_switch.ipynb).
3. **Start a vLLM server** with that model:
   ```bash
   python -m vllm.entrypoints.openai.api_server --model <repo-or-path> --port 8000
   ```
4. **Verify the server** is reachable: `curl http://localhost:8000/v1/models`

New to mellea intrinsics? Start with [`hello_mellea.ipynb`](../quickstart/hello_mellea.ipynb) for a softer walkthrough of each intrinsic in isolation. Full setup details (GPU sizes, HF auth, multi-GPU) are in [`../PREREQUISITES.md`](../PREREQUISITES.md).

**Pipeline steps:**
1. **Guardian** — blocks harmful content first, then out-of-scope queries (not about government services)
2. **Query Rewrite** — disambiguates the query using conversation history
3. **ChromaDB Retrieval** — dense vector search with `granite-embedding-small-english-r2`
4. **Answerability** — exits early if the retrieved documents cannot answer the query
5. **Query Clarification** — asks a follow-up if the retrieved context isn't sufficient
6. **Base Model** — generates the answer grounded in retrieved documents
7. **Citations** — extracts the document spans that support the answer

## 1 · Configuration

In [ ]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../.env"), override=False)
except ImportError:
    pass

# ── vLLM server ───────────────────────────────────────────────────────────────
# URL of the running vLLM OpenAI-compatible endpoint.
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1")

# Model name as reported by GET /v1/models (usually the path/repo used at launch).
VLLM_MODEL_NAME = os.environ.get("VLLM_MODEL_NAME", "ibm-granite/granite-switch-4.1-3b-preview")

# HF Hub repo ID (or local path) to load I/O configs for the embedded adapters.
GRANITE_SWITCH_SOURCE = os.environ.get("GRANITE_SWITCH_SOURCE", VLLM_MODEL_NAME)

# Guardian: which safety criterion to evaluate
GUARDIAN_CRITERIA = "harm"  # harm | social_bias | groundedness | jailbreak | ...

# ── Embedding model (used to build + query ChromaDB) ─────────────────────────
EMBEDDING_MODEL_ID = "ibm-granite/granite-embedding-small-english-r2"

# ── ChromaDB persistence path ─────────────────────────────────────────────────
# Share this directory (zipped) to skip the extraction step entirely.
CHROMA_PATH = "./govt_chroma"

# ── Corpus source (only needed when building the index from scratch) ─────────
# govt.jsonl: 49k government-service passages from IBM mt-rag-benchmark.
GOVT_JSONL_URL  = "https://github.com/IBM/mt-rag-benchmark/raw/main/corpora/passage_level/govt.jsonl.zip"
GOVT_JSONL_PATH = "./govt.jsonl"

# ── Retrieval ─────────────────────────────────────────────────────────────────
TOP_K = 20

# ── Answerability ─────────────────────────────────────────────────────────────
ANSWERABILITY_THRESHOLD = 0.5

print(f"vLLM:      {VLLM_BASE_URL}  ({VLLM_MODEL_NAME})")
print(f"Embedding: {EMBEDDING_MODEL_ID}")
print(f"ChromaDB:  {CHROMA_PATH}")

## 2 · Build or load vector corpus
Data prep is delegated to `scripts/govt_data_loader.py` to keep this notebook focused on the RAG pipeline.

In [ ]:
# Load or build the ChromaDB corpus.
# First run: downloads govt.jsonl.zip from IBM mt-rag-benchmark (49k passages),
# embeds with `ibm-granite/granite-embedding-small-english-r2` into ./govt_chroma.
# Subsequent runs: loads ./govt_chroma instantly.
from tutorials.scripts.govt_data_loader import load_or_build_govt_chroma

chroma_collection = load_or_build_govt_chroma(
    chroma_path        = CHROMA_PATH,
    jsonl_path         = GOVT_JSONL_PATH,
    jsonl_url          = GOVT_JSONL_URL,
    embedding_model_id = EMBEDDING_MODEL_ID,
)


## 3 · Connect to vLLM backend
Registers the Granite Switch embedded adapters from `GRANITE_SWITCH_SOURCE`
so all intrinsics (guardian, RAG, citations) route through the correct control tokens.

In [ ]:
from IPython.display import display, Markdown

from mellea.backends.openai import OpenAIBackend
from mellea.stdlib.context import ChatContext
from mellea.stdlib.components import Document as MelleaDocument
from mellea.stdlib.components.chat import Message as MelleaMessage
import mellea.stdlib.functional as mfuncs

backend = OpenAIBackend(
    model_id=VLLM_MODEL_NAME,
    base_url=VLLM_BASE_URL,
    api_key="unused",
)
backend.register_embedded_adapter_model(GRANITE_SWITCH_SOURCE)
print(f"Backend ready — {backend.list_adapters()}")

## 4 · Functions + conversation state
Run this cell once after loading the backend.

In [ ]:
import json
from mellea.backends import ModelOption
from mellea.stdlib.components.intrinsic import rag
from mellea.stdlib.components.intrinsic.guardian import guardian_check


# ── Guardian criteria ─────────────────────────────────────────────────────────
# Scope check: positive definition — message should be about government services.
GUARDIAN_SCOPE_CRITERIA = (
    "Governmental services content refers to messages concerning services "
    "that are provided, administered, funded, or regulated by a government "
    "agency at any level — federal, state, local, or municipal. This "
    "includes taxes and tax filings, public benefits (such as social "
    "security, disability benefits, unemployment, food assistance, Medicaid), "
    "permits and licenses, voting and elections, immigration, public healthcare "
    "programs, housing assistance, veterans affairs, public records, "
    "court and legal processes, and direct interactions with any "
    "government office or program."
)

# Late instruction appended to the user message at generation time.
# Placed after history + documents to preserve KV cache prefix sharing.
GENERATION_INSTRUCTION = (
    "Answer concisely and directly based only on the provided documents. "
    "Do not repeat the question or add unnecessary preamble."
)


def _is_clear(clarification):
    """QC returns 'CLEAR' when no clarification is needed; accept prefix variants like 'CLEARLY'."""
    return clarification.strip().upper().startswith("CLEAR")


# ── Full pipeline ─────────────────────────────────────────────────────────────
def run_pipeline(query, history=None):
    """Run the full RAG pipeline for one user turn. Returns a result dict."""
    if history is None:
        history = []
    r = {"query": query, "blocked": False, "unanswerable": False, "needs_clarification": False}

    # Build a ChatContext from stored history — reused across all intrinsic calls.
    ctx = ChatContext()
    for m in history:
        docs = [MelleaDocument(doc_id=str(i), text=t)
                for i, t in enumerate(m.get("documents") or [])] or None
        ctx = ctx.add(MelleaMessage(m["role"], m["content"], documents=docs))
    ctx_with_query = ctx.add(MelleaMessage("user", query))

    # [1a] Harm check — run first so harmful content is flagged as harmful - harm criteria from mellea dictionary
    r["guardian_harm_score"] = guardian_check(ctx_with_query, backend, "harm", target_role="user")
    if r["guardian_harm_score"] >= 0.5:
        r["blocked"] = True
        r["block_reason"] = f"Harmful content detected (score={r['guardian_harm_score']:.3f})"
        return r

    # [1b] Scope check — is the query about government services? criteria provided to mellea by caller 
    r["guardian_scope_score"] = guardian_check(ctx_with_query, backend, GUARDIAN_SCOPE_CRITERIA, target_role="user")
    if r["guardian_scope_score"] < 0.5:
        r["blocked"] = True
        r["block_reason"] = f"Out of scope — not a government services topic (score={r['guardian_scope_score']:.3f})"
        return r

    # [2] Rewrite query using conversation history.
    r["rewritten_query"] = rag.rewrite_question(query, ctx, backend)

    # [3] Retrieve candidate documents from ChromaDB.
    r["documents"] = chroma_collection.query(query_texts=[r["rewritten_query"]], n_results=TOP_K)["documents"][0]
    mellea_docs = [MelleaDocument(doc_id=str(i), text=t) for i, t in enumerate(r["documents"])]

    # [4] Answerability — can the retrieved docs answer the query?
    r["answerability"] = rag.check_answerability(r["rewritten_query"], mellea_docs, ctx, backend)
    if r["answerability"] == "unanswerable":
        r["unanswerable"] = True
        return r

    # [5] Clarification — if docs are not enough, ask a follow-up question.
    r["clarification"] = rag.clarify_query(r["rewritten_query"], mellea_docs, ctx, backend)
    if not _is_clear(r["clarification"]):
        r["needs_clarification"] = True
        return r

    # [6] Base model — generate the answer grounded in the retrieved documents.
    prompted_query = r["rewritten_query"] + "\n\n" + GENERATION_INSTRUCTION
    out, _ = mfuncs.act(
        MelleaMessage("user", prompted_query, documents=mellea_docs),
        ctx, backend,
        model_options={ModelOption.TEMPERATURE: 0.0},
    )
    r["answer"] = str(out)

    # [7] Citations — map answer spans to supporting document passages.
    r["citations"] = rag.find_citations(r["answer"], mellea_docs, ctx_with_query, backend) if mellea_docs else []
    return r


# ── Display: final answer only ────────────────────────────────────────────────
def show_answer(r):
    lines = [f"**Q:** {r['query']}", "---"]
    if r.get("blocked"):
        lines.append(f"⛔ **BLOCKED** — {r['block_reason']}")
    elif r.get("unanswerable"):
        lines.append(
            f"🔍 **Not in corpus** — `answerability={r['answerability']}`\n\n"
            f"> I don't have enough information in my knowledge base to answer that."
        )
    elif r.get("needs_clarification"):
        lines.append(f"❓ **Clarification needed:**\n\n> {r['clarification']}")
    else:
        lines.append(f"**A:** {r.get('answer', '')}")
    display(Markdown("\n\n".join(lines)))


# ── Conversation: stateful chat on top of run_pipeline ───────────────────────
class Conversation:
    """A single conversation instance. Owns a history and knows how to ask
    on top of the stateless `run_pipeline`."""

    def __init__(self):
        self.history = []

    def ask(self, query):
        print(f"[turn {len(self.history)//2 + 1}  |  history: {len(self.history)} msg(s)]")
        r = run_pipeline(query, self.history)
        show_answer(r)

        if r["blocked"]:
            return r  # blocked turns are not recorded

        if r["unanswerable"]:
            reply = "I don't have enough information in my knowledge base to answer that."
        elif r["needs_clarification"]:
            reply = r["clarification"]
        else:
            reply = r.get("answer", "")

        self.history.append({"role": "user",      "content": query, "documents": r.get("documents")})
        self.history.append({"role": "assistant", "content": reply})
        print(f"→ history now has {len(self.history)} message(s)")
        return r


print("✅ run_pipeline + Conversation ready.")


In [ ]:
# Display helper functions : 
#   show_intermediates: step-by-step intermediates display for full examination of the rag pipeline
#   show_history: for full conversation history view 
import logging, warnings

def show_intermediates(r):
    md = []
    md.append("---")
    md.append(f"### Intermediates — *{r['query']}*")
    md.append("---")

    harm_score = r.get("guardian_harm_score", 0)
    harm_badge = "🟢 safe" if harm_score < 0.5 else "🔴 harmful"
    md.append(f"**[1a] Guardian — Harm** — {harm_badge} &nbsp;&nbsp; `score={harm_score:.3f}` &nbsp;&nbsp; (full-conversation eval)")

    if r.get("blocked") and "Harmful" in r.get("block_reason", ""):
        md.append(f"\n> ⛔ **BLOCKED:** {r['block_reason']}")
        display(Markdown("\n\n".join(md)))
        return

    scope_score = r.get("guardian_scope_score", 0)
    scope_badge = "🟢 in-scope" if scope_score >= 0.5 else "🔴 out-of-scope"
    md.append(f"\n**[1b] Guardian — Scope** — {scope_badge} &nbsp;&nbsp; `score={scope_score:.3f}`")

    if r.get("blocked"):
        md.append(f"\n> ⛔ **BLOCKED:** {r['block_reason']}")
        display(Markdown("\n\n".join(md)))
        return

    md.append(f"\n**[2] Query Rewrite**\n\n"
              f"| | |\n|---|---|\n"
              f"| original | {r['query']} |\n"
              f"| rewritten | {r.get('rewritten_query')} |")

    docs = r.get("documents", [])
    md.append(f"\n**[3] ChromaDB Retrieval** — {len(docs)} doc(s) (top {TOP_K}, cosine sim)")
    if docs:
        md.append(f"\n<details><summary>📚 Show all {len(docs)} documents</summary>\n")
        for i, d in enumerate(docs):
            md.append(f"<details><summary>📄 Document {i+1}</summary>\n\n```\n{d}\n```\n\n</details>\n")
        md.append("</details>")

    answerability = r.get("answerability")
    if answerability is not None:
        badge = "✅ answerable" if not r.get("unanswerable") else "🔍 unanswerable"
        md.append(f"\n**[4] Answerability** — {badge} &nbsp;&nbsp; `verdict={answerability}`")
    if r.get("unanswerable"):
        display(Markdown("\n\n".join(md)))
        return

    clar  = r.get("clarification", "")
    badge = "✅ CLEAR" if _is_clear(clar) else "❓ needs clarification"
    md.append(f"\n**[5] Clarification** — {badge}")
    if r.get("needs_clarification"):
        md.append(f"\n> {clar}")
        display(Markdown("\n\n".join(md)))
        return

    ans = r.get("answer", "")
    md.append(f"\n**[6] Answer** — {len(ans)} chars\n\n> {ans}")

    citations = r.get("citations", [])
    md.append(f"\n**[7] Citations** — {len(citations)} found")
    if citations:
        md.append(f"\n<details><summary>🔖 Show citations JSON</summary>\n\n```json\n{json.dumps(citations, indent=2)}\n```\n\n</details>")
    else:
        md.append("\n*(none)*")

    display(Markdown("\n\n".join(md)))
    
def show_history(conv):
    """Render a Conversation's history as formatted Markdown."""
    if not conv.history:
        display(Markdown("*(conversation history is empty)*"))
        return
    md = ["---", f"### Conversation history — {len(conv.history)//2} turn(s)", "---"]
    for m in conv.history:
        role = "👤 **User**" if m["role"] == "user" else "🤖 **Assistant**"
        doc_note = f" *({len(m['documents'])} docs)*" if m.get("documents") else ""
        md.append(f"{role}{doc_note}\n\n> {m['content']}")
    display(Markdown("\n\n".join(md)))

#for cleaner display of questions results
# Silence mellea's INFO/WARNING chatter (TemplateRepresentation, "Tools for call", SUCCESS)
logging.getLogger("mellea").setLevel(logging.ERROR)
# Catch any stray warnings the logger doesn't cover
warnings.filterwarnings("ignore", message=".*TemplateRepresentation.*")

## 5 · Queries
Each cell is one turn. History accumulates automatically.
- `ask(query)` — run pipeline + show answer + update history
- `show_intermediates(r)` — see step-by-step breakdown for any result
- `show_history()` — print full conversation so far

### Reference: `show_intermediates(r)`

Reveals every step the pipeline took for a given result. Pass any result variable — `r1`, `r2`, `r3`, `r4`, `r5`.

| Step | What you'll see |
|------|----------------|
| **[1] Guardian** | Safety verdict (`safe` / `flagged`) and raw score |
| **[2] Query Rewrite** | Original query vs. the rewritten version (uses conversation history to disambiguate) |
| **[3] ChromaDB Retrieval** | Number of documents retrieved; each doc is collapsible |
| **[4] Answerability** | Likelihood score (0–1) that the retrieved documents can answer the query; exits early if below threshold |
| **[5] Clarification** | `CLEAR` if the model had enough context to answer; otherwise the follow-up question it would ask |
| **[6] Answer** | Full model response |
| **[7] Citations** | Document spans the model grounded its answer in (JSON) |

In [ ]:
conv = Conversation()
# Q1 — happy path: specific question, CLEAR, full answer with citations
r1 = conv.ask("How do I check the status of my federal tax refund?")
show_intermediates(r1)

In [ ]:
# Q2 — ambiguous: multiple disability programs exist; pipeline asks which one
r2 = conv.ask("I want to apply for disability benefits")
show_intermediates(r2)

In [ ]:
# Q3 — resolves clarification: query rewrite uses history to reconstruct full question
r3 = conv.ask("State Disability Insurance")
show_intermediates(r3)

In [ ]:
# Q4 — unanswerable: natural follow-up from Q3 but individual benefit amounts aren't in a static corpus
r4 = conv.ask("How much would I receive in disability benefits?")
show_intermediates(r4)

In [ ]:
# Q5 — out of scope: weather is not a government services topic; guardian blocks it
r5 = conv.ask("What's the weather in New York tomorrow?")
show_intermediates(r5)

In [ ]:
# Q6 — guardian blocks request to forge government documents
r6 = conv.ask("How do I forge a government ID?")
show_intermediates(r6)

In [ ]:
show_history(conv)

## Next steps

- **Adapt this to your own app.** `run_pipeline(query, history)` is stateless and self-contained — copy it as a starting point and swap in your own corpus, scope criteria, or generation instruction.
- **Use different adapters.** See [`../how-to/bring_your_own_adapter.md`](../how-to/bring_your_own_adapter.md) to train a custom adapter and fold it into a Granite Switch checkpoint.
- **Compose your own checkpoint.** [`03_compose_granite_switch.ipynb`](./03_compose_granite_switch.ipynb) walks through building a model from the IBM adapter libraries.
- **Go deeper on mellea.** [Mellea on GitHub](https://github.com/generative-computing/mellea) — the intrinsics framework powering this notebook.
- **Browse more adapters.** [IBM Granite adapter libraries](https://huggingface.co/collections/ibm-granite/granite-libraries) — RAG, core, and guardian libraries on Hugging Face.